In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import json
from datetime import datetime

## Set up notebooks

In [ ]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [ ]:
adata = sc.read_h5ad('data/gut_data/maxfuse_checkpoints/gut_hs_XeniumHealthyColon_maxfuse_imputed_adata_07042025_155610.h5ad')

In [ ]:
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].cat.add_categories(['Arterial capillary',
                                                                                  'Gamma delta T cells'])
adata.obs.loc[(adata.obs['C_scANVI'] == 'Adult Glia'), 'C_scANVI'] = 'Glial cells'
adata.obs.loc[(adata.obs['C_scANVI'] == 'arterial capillary'), 'C_scANVI'] = 'Arterial capillary'
adata.obs.loc[(adata.obs['C_scANVI'] == 'gdT'), 'C_scANVI'] = 'Gamma delta T cells'
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].replace('Enterocyte', 'Colonocyte')
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].replace('Gamma delta T cells', 'CD8 T')
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].replace('Tregs', 'CD4 T')
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].replace('Mesothelium', 'Fibroblasts')
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].replace('Lymphatics', 'LEC')
adata.obs['C_scANVI'] = adata.obs['C_scANVI'].cat.remove_unused_categories()

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_39005/228337369.py:6: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_39005/228337369.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_39005/228337369.py:8: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_

* Update niches 

In [ ]:
adata_nichecompass = sc.read_h5ad('data/NicheCompass/artifacts/single_sample/26032025_173236/model/xenium_adult_colon_hs_NicheCompass_AM_26032025_173236_raw.h5ad')

In [ ]:
adata.obs['latent_leiden_0.4'] = adata_nichecompass.obs['latent_leiden_0.4'].copy()

In [ ]:
adata.obs['latent_leiden_0.4'].value_counts()

latent_leiden_0.4
0     31855
1     26695
2     25249
3     22345
4     22266
5     20983
6     20849
7     18868
8     17103
9     16214
10    13495
11    13233
12    12231
13     9477
14     1953
15     1221
Name: count, dtype: int64

In [ ]:
adata.obsm['X_umap'] = adata_nichecompass.obsm['X_umap'].copy()

In [ ]:
adata.uns['spatial'] = adata_nichecompass.uns['spatial'].copy()
adata.obsm['spatial'] = adata_nichecompass.obsm['spatial'].copy()

In [ ]:
sc.pp.normalize_total(adata_nichecompass)
sc.pp.log1p(adata_nichecompass)

In [ ]:
assert np.all(adata.obs_names == adata_nichecompass.obs_names), "Cell indices don't match!"

In [ ]:
adata_nichecompass.var['genes_type']='original'
adata.var['genes_type']='imputed'

In [ ]:
del adata_nichecompass.varm

In [ ]:
full_obs = adata.obs.copy()

In [ ]:
adata = ad.concat(
    [adata, adata_nichecompass], 
    axis=1,  # 1 is for variables/genes axis
    join='outer',  # Keep all genes from both objects
    merge='same'   # Only merge observation (cell) annotations that are the same
)

In [ ]:
adata.obs = full_obs

In [ ]:
adata

AnnData object with n_obs × n_vars = 274037 × 10281
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'SLPI_ligand_receptor_target_gene_GP', 'CXCL14_ligand_receptor_target_gene_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'IL1B_ligand_receptor_target_gene_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CDH1_ligand_receptor_target_gene_GP', 'TNXB_ligand_receptor_target_gene_GP', 'CLU_ligand_receptor_target_gene_GP', 'TFF1_ligand_receptor_target_gene_GP', 'CCL11_ligand_receptor_target_gene_GP', 'ROBO1_ligand_receptor_target_gene_GP', 'NRG3_ligand_recepto

In [ ]:
del adata_nichecompass

In [ ]:
adata.write_h5ad(f"data/gut_data/maxfuse_checkpoints/gut_hs_XeniumHealthyColon_maxfuse_imputed_and_originalcounts_updated_niches_{timestamp}_log.h5ad")

... storing 'genes_type' as categorical
... storing 'gene_ids' as categorical
